In [1]:
import pandas as pd
import numpy as np

In [2]:
# Open our data + basic info

# utf-8 causes decoding error
df = pd.read_csv('C:/my pet projects/shark_attacks/data/raw/shark_attacks_raw.csv',encoding='latin1')
print(df.shape)
print(df.info())    

(25723, 24)
<class 'pandas.DataFrame'>
RangeIndex: 25723 entries, 0 to 25722
Data columns (total 24 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Case Number             8702 non-null   str    
 1   Date                    6302 non-null   str    
 2   Year                    6300 non-null   float64
 3   Type                    6298 non-null   str    
 4   Country                 6252 non-null   str    
 5   Area                    5847 non-null   str    
 6   Location                5762 non-null   str    
 7   Activity                5758 non-null   str    
 8   Name                    6092 non-null   str    
 9   Sex                     5737 non-null   str    
 10  Age                     3471 non-null   str    
 11  Injury                  6274 non-null   str    
 12  Fatal (Y/N)             5763 non-null   str    
 13  Time                    2948 non-null   str    
 14  Species                 3464 non-null

In [ ]:
# Percent of empty per column
missing_percent = df.isnull().mean().sort_values(ascending=False) * 100
print("Percentage of missing values per column:")
print(missing_percent)

In [ ]:
# Delete empty columns
df = df.drop(columns=['Unnamed: 22', 'Unnamed: 23'])

#Accuracy down to the day of the month is unlikely to be necessary; we'll single out the month, as statistics on attacks depending on the season may be of interest
df['Month'] = pd.to_datetime(df['Date'], errors='coerce').dt.month

# Delete date
df = df.drop(columns=['Date'])

print(f"After deleting empty columns: {df.shape}")

In [ ]:
necessary_columns = ['Fatal (Y/N)','Month', 'Case Number','Country','Area', 'Type', 'Activity']

# Delete rows where all the main characteristics are empty

df = df.dropna(subset=necessary_columns, how='all')
print(f"After deleting rows with empty main characteristics: {df.shape}")

# Percent of empty per column after cleaning
print(df.isnull().mean().sort_values(ascending=False) * 100)

In [ ]:
# Fatal go from Y/N represent to binar 1/0 
def fatal_to_binary(value):
    if pd.isna(value):
        return None
    if value == "Y":
        return 1
    if value == "N":
        return 0
    else:
        return None

df['Fatal'] = df['Fatal (Y/N)'].apply(fatal_to_binary)


print(df[['Fatal (Y/N)', 'Fatal']].head(10))
print(df['Fatal'].value_counts(dropna=False))

# Get rid of Fatal(Y/N) because now we have clear binary view
df = df.drop(columns=['Fatal (Y/N)'])


In [ ]:
# Rename column 'Species ' to 'Species'
df = df.rename(columns={'Species ': 'Species'})



# Sex to binary where f = 1, m = 0

def sex_to_binary(value):
    val = value.lower()
    if pd.isna(val):
        return None
    if "f" in val:
        return 1
    if "m" in val:
        return 0
    else:
        return None

df['Sex'] = df['Sex '].apply(sex_to_binary)


print(df[['Sex', 'Sex ']].head(10))
print(df['Sex'].value_counts(dropna=False))

# Get rid of Fatal(Y/N) because now we have clear binary view
df = df.drop(columns=['Sex '])


In [ ]:
# text columns to make lower case plus clear excess spaces
text_columns = ['Case Number', 'Type', 'Country', 'Area', 'Location', 'Activity', 'Name', 'Injury', 'Species', "Time"]

for column in text_columns:
        # Convert to string, lowercase, strip whitespace
        df[column] = df[column].astype(str).str.lower().str.strip()
        # Replace common empty string representations with NaN
        df[column] = df[column].replace(['nan', 'none', '', 'unknown'], None)
